In [1]:
!pip install -q pandas numpy scikit-learn joblib lazypredict

In [2]:
import warnings
warnings.filterwarnings("ignore")

from pathlib import Path
import joblib
import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet, BayesianRidge, HuberRegressor
from sklearn.neighbors import KNeighborsRegressor
from sklearn.svm import SVR, LinearSVR
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import (
    RandomForestRegressor,
    ExtraTreesRegressor,
    GradientBoostingRegressor,
    HistGradientBoostingRegressor,
    AdaBoostRegressor,
    BaggingRegressor
)
from sklearn.neural_network import MLPRegressor

In [3]:
from pathlib import Path
import pandas as pd

DATA_DIR = Path("..") / "data" / "processed"

TRAIN_PATH = DATA_DIR / "train.csv"
VAL_PATH = DATA_DIR / "val.csv"
TEST_PATH = DATA_DIR / "test.csv"

print("Current working directory:", Path.cwd())

print("TRAIN_PATH:", TRAIN_PATH.resolve(), TRAIN_PATH.exists())
print("VAL_PATH:", VAL_PATH.resolve(), VAL_PATH.exists())
print("TEST_PATH:", TEST_PATH.resolve(), TEST_PATH.exists())

train_df = pd.read_csv(TRAIN_PATH)
val_df = pd.read_csv(VAL_PATH)
test_df = pd.read_csv(TEST_PATH)

print("Train:", train_df.shape)
print("Val:", val_df.shape)
print("Test:", test_df.shape)

display(train_df.head())

Current working directory: c:\Users\Xuanuio\Desktop\Deadline\Kho_DL\Data_Minning\Minning\Model
TRAIN_PATH: C:\Users\Xuanuio\Desktop\Deadline\Kho_DL\Data_Minning\Minning\Data\processed\train.csv True
VAL_PATH: C:\Users\Xuanuio\Desktop\Deadline\Kho_DL\Data_Minning\Minning\Data\processed\val.csv True
TEST_PATH: C:\Users\Xuanuio\Desktop\Deadline\Kho_DL\Data_Minning\Minning\Data\processed\test.csv True
Train: (3385, 16)
Val: (1128, 16)
Test: (1129, 16)


,country_name,country_code,year,population,pop_growth,life_expectancy,gdp_growth,sanitation,electricity,water_access,co2_emissions,labor_force,is_covid,log1p_gdp_per_capita,electricity_water_interaction,basic_services_gap
0,Israel,ISR,2006.0,-0.203895,0.304521,80.553659,0.385675,1.146906,0.626003,0.741158,0.466569,0.117087,0,0.867560,0.759269,-0.934425
1,Qatar,QAT,2010.0,-0.245809,-0.486913,79.444000,2.744363,1.322539,0.626003,0.736250,4.256240,2.561888,0,1.644051,0.756579,-1.005932
2,Ghana,GHA,2022.0,-0.002721,0.399084,65.246000,0.082754,-1.289462,0.098067,0.039351,-0.442584,0.238940,0,-0.601728,-0.040621,0.482423
3,Tanzania,TZA,2008.0,0.069912,0.884395,59.896000,0.399970,-1.455111,-2.509724,-2.736538,-0.503541,2.642554,0,-1.359459,-2.267199,2.382605
4,Norway,NOR,2022.0,-0.216204,-0.234645,82.509756,-0.011425,0.903913,0.626003,0.741158,0.333926,0.471697,0,1.870898,0.759269,-0.833726


In [4]:
# =========================
# CONFIG
# =========================

DATA_DIR = Path("data/processed")
OUTPUT_DIR = Path("model_outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

TARGET_CANDIDATES = {
    "life_expectancy",
    "life_expectancy_years",
    "lifeexpectancy",
    "target",
    "y"
}

RANDOM_STATE = 42

OUTPUT_DIR = Path("model_outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

In [5]:
# =========================
# LOAD DATA
# =========================

train_df = pd.read_csv(TRAIN_PATH)
val_df = pd.read_csv(VAL_PATH)
test_df = pd.read_csv(TEST_PATH)

print("Train shape:", train_df.shape)
print("Val shape:", val_df.shape)
print("Test shape:", test_df.shape)

display(train_df.head())

Train shape: (3385, 16)
Val shape: (1128, 16)
Test shape: (1129, 16)


,country_name,country_code,year,population,pop_growth,life_expectancy,gdp_growth,sanitation,electricity,water_access,co2_emissions,labor_force,is_covid,log1p_gdp_per_capita,electricity_water_interaction,basic_services_gap
0,Israel,ISR,2006.0,-0.203895,0.304521,80.553659,0.385675,1.146906,0.626003,0.741158,0.466569,0.117087,0,0.867560,0.759269,-0.934425
1,Qatar,QAT,2010.0,-0.245809,-0.486913,79.444000,2.744363,1.322539,0.626003,0.736250,4.256240,2.561888,0,1.644051,0.756579,-1.005932
2,Ghana,GHA,2022.0,-0.002721,0.399084,65.246000,0.082754,-1.289462,0.098067,0.039351,-0.442584,0.238940,0,-0.601728,-0.040621,0.482423
3,Tanzania,TZA,2008.0,0.069912,0.884395,59.896000,0.399970,-1.455111,-2.509724,-2.736538,-0.503541,2.642554,0,-1.359459,-2.267199,2.382605
4,Norway,NOR,2022.0,-0.216204,-0.234645,82.509756,-0.011425,0.903913,0.626003,0.741158,0.333926,0.471697,0,1.870898,0.759269,-0.833726


In [6]:
# =========================
# FIND TARGET COLUMN
# =========================

def normalize_colname(col):
    return str(col).strip().lower().replace(" ", "_").replace("-", "_")

target_col = None
for col in train_df.columns:
    if normalize_colname(col) in TARGET_CANDIDATES:
        target_col = col
        break

if target_col is None:
    raise ValueError(
        "Không tìm thấy cột target. Các cột hiện có là:\n"
        + str(list(train_df.columns))
    )

print("Target column:", target_col)

Target column: life_expectancy


In [7]:
# =========================
# SPLIT X, y
# =========================

def split_xy(df, target):
    X = df.drop(columns=[target])
    y = df[target]
    return X, y

X_train, y_train = split_xy(train_df, target_col)
X_val, y_val = split_xy(val_df, target_col)
X_test, y_test = split_xy(test_df, target_col)

# Đảm bảo val/test có đúng thứ tự cột như train
X_val = X_val[X_train.columns]
X_test = X_test[X_train.columns]

print("X_train:", X_train.shape)
print("X_val:", X_val.shape)
print("X_test:", X_test.shape)

X_train: (3385, 15)
X_val: (1128, 15)
X_test: (1129, 15)


In [8]:
# =========================
# PREPROCESSING
# =========================

numeric_cols = X_train.select_dtypes(include=[np.number]).columns.tolist()
categorical_cols = X_train.select_dtypes(exclude=[np.number]).columns.tolist()

print("Numeric columns:", numeric_cols)
print("Categorical columns:", categorical_cols)

def make_onehot_encoder():
    try:
        return OneHotEncoder(handle_unknown="ignore", sparse_output=False)
    except TypeError:
        return OneHotEncoder(handle_unknown="ignore", sparse=False)

numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", make_onehot_encoder())
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_cols),
        ("cat", categorical_transformer, categorical_cols)
    ],
    remainder="drop"
)

Numeric columns: ['year', 'population', 'pop_growth', 'gdp_growth', 'sanitation', 'electricity', 'water_access', 'co2_emissions', 'labor_force', 'is_covid', 'log1p_gdp_per_capita', 'electricity_water_interaction', 'basic_services_gap']
Categorical columns: ['country_name', 'country_code']


In [9]:
# =========================
# MODELS
# =========================

models = {
    "DummyMean": DummyRegressor(strategy="mean"),

    "LinearRegression": LinearRegression(),
    "Ridge": Ridge(alpha=1.0, random_state=RANDOM_STATE),
    "Lasso": Lasso(alpha=0.001, random_state=RANDOM_STATE, max_iter=10000),
    "ElasticNet": ElasticNet(alpha=0.001, l1_ratio=0.5, random_state=RANDOM_STATE, max_iter=10000),
    "BayesianRidge": BayesianRidge(),
    "HuberRegressor": HuberRegressor(max_iter=1000),

    "KNN": KNeighborsRegressor(n_neighbors=5),

    "DecisionTree": DecisionTreeRegressor(
        random_state=RANDOM_STATE,
        max_depth=None
    ),

    "RandomForest": RandomForestRegressor(
        n_estimators=300,
        random_state=RANDOM_STATE,
        n_jobs=-1
    ),

    "ExtraTrees": ExtraTreesRegressor(
        n_estimators=300,
        random_state=RANDOM_STATE,
        n_jobs=-1
    ),

    "GradientBoosting": GradientBoostingRegressor(
        random_state=RANDOM_STATE
    ),

    "HistGradientBoosting": HistGradientBoostingRegressor(
        random_state=RANDOM_STATE
    ),

    "AdaBoost": AdaBoostRegressor(
        n_estimators=200,
        random_state=RANDOM_STATE
    ),

    "Bagging": BaggingRegressor(
        n_estimators=200,
        random_state=RANDOM_STATE,
        n_jobs=-1
    ),

    "SVR_RBF": SVR(kernel="rbf", C=10, epsilon=0.1),

    "LinearSVR": LinearSVR(
        random_state=RANDOM_STATE,
        max_iter=10000
    ),

    "MLP": MLPRegressor(
        hidden_layer_sizes=(128, 64),
        activation="relu",
        solver="adam",
        max_iter=1000,
        random_state=RANDOM_STATE
    )
}

In [10]:
# =========================
# EVALUATION FUNCTIONS
# =========================

def rmse(y_true, y_pred):
    return np.sqrt(mean_squared_error(y_true, y_pred))

def evaluate_model(model_name, pipeline, X, y, split_name):
    pred = pipeline.predict(X)
    return {
        "model": model_name,
        "split": split_name,
        "MAE": mean_absolute_error(y, pred),
        "RMSE": rmse(y, pred),
        "R2": r2_score(y, pred)
    }

In [11]:
# =========================
# TRAIN + VALIDATE + TEST
# =========================

all_results = []
trained_models = {}

for name, model in models.items():
    print(f"Training: {name}")

    pipeline = Pipeline(steps=[
        ("preprocess", preprocessor),
        ("model", model)
    ])

    try:
        pipeline.fit(X_train, y_train)

        val_result = evaluate_model(name, pipeline, X_val, y_val, "val")
        test_result = evaluate_model(name, pipeline, X_test, y_test, "test")

        all_results.append(val_result)
        all_results.append(test_result)

        trained_models[name] = pipeline

    except Exception as e:
        print(f"Model {name} lỗi: {e}")

results_df = pd.DataFrame(all_results)

ranked_df = (
    results_df[results_df["split"] == "val"]
    .sort_values("RMSE", ascending=True)
    .reset_index(drop=True)
)

display(ranked_df)

Training: DummyMean
Training: LinearRegression
Training: Ridge
Training: Lasso
Training: ElasticNet
Training: BayesianRidge
Training: HuberRegressor
Training: KNN
Training: DecisionTree
Training: RandomForest
Training: ExtraTrees
Training: GradientBoosting
Training: HistGradientBoosting
Training: AdaBoost
Training: Bagging
Training: SVR_RBF
Training: LinearSVR
Training: MLP


,model,split,MAE,RMSE,R2
0,ExtraTrees,val,0.603863,1.191633,0.980076
1,MLP,val,0.770935,1.258666,0.977772
2,SVR_RBF,val,0.644627,1.451299,0.970447
3,Bagging,val,0.957757,1.504182,0.968254
4,RandomForest,val,0.959124,1.509743,0.968019
5,LinearRegression,val,0.982670,1.555984,0.966030
6,BayesianRidge,val,0.992844,1.580033,0.964972
7,HuberRegressor,val,0.965650,1.611645,0.963556
8,Ridge,val,1.070449,1.693520,0.959759
9,Lasso,val,1.117845,1.736369,0.957697


In [12]:
# =========================
# SHOW FULL RESULT TABLE
# =========================

full_table = (
    results_df
    .pivot(index="model", columns="split", values=["MAE", "RMSE", "R2"])
)

display(full_table)

results_df.to_csv(OUTPUT_DIR / "all_model_results.csv", index=False)
ranked_df.to_csv(OUTPUT_DIR / "ranked_models_by_val_rmse.csv", index=False)

print("Đã lưu:")
print(OUTPUT_DIR / "all_model_results.csv")
print(OUTPUT_DIR / "ranked_models_by_val_rmse.csv")

MAE                RMSE                  R2  \
split                     test       val      test       val      test   
model                                                                    
AdaBoost              3.736140  3.524966  4.426238  4.236174  0.735794   
Bagging               0.927533  0.957757  1.443325  1.504182  0.971907   
BayesianRidge         1.024265  0.992844  1.572682  1.580033  0.966645   
DecisionTree          1.071790  1.164643  1.880304  2.120363  0.952321   
DummyMean             6.888276  6.771542  8.619726  8.442323 -0.001982   
ElasticNet            1.163206  1.161828  1.753643  1.811452  0.958528   
ExtraTrees            0.534260  0.603863  0.931494  1.191633  0.988299   
GradientBoosting      2.044190  2.020569  2.611030  2.615922  0.908062   
HistGradientBoosting  1.260846  1.296844  1.749806  1.899563  0.958709   
HuberRegressor        0.999099  0.965650  1.660489  1.611645  0.962817   
KNN                   1.029989  0.988753  1.879659  1.772613  0.952354   
Lasso                 1.119814  1.117845  1.697079  1.736369  0.961160   
LinearRegression      1.020545  0.982670  1.563357  1.555984  0.967040   
LinearSVR             1.117542  1.090920  1.863584  1.837902  0.953165   
MLP                   0.771103  0.770935  1.519476  1.258666  0.968864   
RandomForest          0.929532  0.959124  1.459800  1.509743  0.971262   
Ridge                 1.083102  1.070449  1.655164  1.693520  0.963055   
SVR_RBF               0.653979  0.644627  1.420520  1.451299  0.972788   

                                
split                      val  
model                           
AdaBoost              0.748214  
Bagging               0.968254  
BayesianRidge         0.964972  
DecisionTree          0.936918  
DummyMean            -0.000017  
ElasticNet            0.953960  
ExtraTrees            0.980076  
GradientBoosting      0.903986  
HistGradientBoosting  0.949372  
HuberRegressor        0.963556  
KNN                   0.955913  
Lasso                 0.957697  
LinearRegression      0.966030  
LinearSVR             0.952605  
MLP                   0.977772  
RandomForest          0.968019  
Ridge                 0.959759  
SVR_RBF               0.970447

Đã lưu:
model_outputs\all_model_results.csv
model_outputs\ranked_models_by_val_rmse.csv


In [13]:
# =========================
# BEST MODEL
# =========================

best_model_name = ranked_df.iloc[0]["model"]
best_pipeline = trained_models[best_model_name]

print("Best model:", best_model_name)

best_val_pred = best_pipeline.predict(X_val)
best_test_pred = best_pipeline.predict(X_test)

print("\nValidation:")
print("MAE :", mean_absolute_error(y_val, best_val_pred))
print("RMSE:", rmse(y_val, best_val_pred))
print("R2  :", r2_score(y_val, best_val_pred))

print("\nTest:")
print("MAE :", mean_absolute_error(y_test, best_test_pred))
print("RMSE:", rmse(y_test, best_test_pred))
print("R2  :", r2_score(y_test, best_test_pred))

Best model: ExtraTrees

Validation:
MAE : 0.6038633857608281
RMSE: 1.1916331365114692
R2  : 0.9800763538243347

Test:
MAE : 0.534259632773101
RMSE: 0.9314944773536821
R2  : 0.9882987196783147


In [14]:
# =========================
# SAVE BEST MODEL
# =========================

model_path = OUTPUT_DIR / f"best_model_{best_model_name}.joblib"
joblib.dump(best_pipeline, model_path)

print("Đã lưu best model tại:", model_path)

Đã lưu best model tại: model_outputs\best_model_ExtraTrees.joblib


In [15]:
# =========================
# SAVE TEST PREDICTIONS
# =========================

test_predictions = test_df.copy()
test_predictions["prediction"] = best_test_pred
test_predictions["error"] = test_predictions[target_col] - test_predictions["prediction"]
test_predictions["abs_error"] = test_predictions["error"].abs()

pred_path = OUTPUT_DIR / "test_predictions.csv"
test_predictions.to_csv(pred_path, index=False)

display(test_predictions.head())

print("Đã lưu dự đoán test tại:", pred_path)

,country_name,country_code,year,population,pop_growth,life_expectancy,gdp_growth,sanitation,electricity,water_access,co2_emissions,labor_force,is_covid,log1p_gdp_per_capita,electricity_water_interaction,basic_services_gap,prediction,error,abs_error
0,Kenya,KEN,2007.0,0.034956,1.082440,59.407,0.596263,-1.022491,-1.914467,-2.072459,-0.491200,1.235063e+00,0,-1.221453,-1.951436,1.777327,59.509857,-0.102857,0.102857
1,Togo,TGO,2014.0,-0.200667,0.779975,58.766,0.391004,-1.596279,-1.297952,-1.638118,-0.494491,-2.883538e-01,0,-1.209938,-1.558974,1.639874,59.162587,-0.396587,0.396587
2,Chile,CHL,2016.0,-0.117448,-0.041765,80.295,-0.263164,1.427949,0.626003,0.639266,0.000845,1.103565e-01,0,0.549242,0.703440,-1.024373,79.772205,0.522795,0.522795
3,Saudi Arabia,SAU,2012.0,-0.056533,1.814487,76.090,0.412641,0.891307,0.626003,0.686721,1.684403,-5.687268e-01,0,1.022170,0.729441,-0.814332,77.217174,-1.127174,1.127174
4,San Marino,SMR,2005.0,-0.258044,0.001024,81.926,-0.149859,1.354057,0.626003,0.741158,-0.267294,7.137836e-16,0,1.488700,0.759269,-1.020271,81.753078,0.172922,0.172922


Đã lưu dự đoán test tại: model_outputs\test_predictions.csv


In [16]:
# =========================
# FEATURE IMPORTANCE
# Chỉ chạy được với model dạng tree như RandomForest, ExtraTrees, GradientBoosting...
# =========================

final_estimator = best_pipeline.named_steps["model"]

if hasattr(final_estimator, "feature_importances_"):
    feature_names = best_pipeline.named_steps["preprocess"].get_feature_names_out()

    importance_df = pd.DataFrame({
        "feature": feature_names,
        "importance": final_estimator.feature_importances_
    }).sort_values("importance", ascending=False)

    importance_path = OUTPUT_DIR / "feature_importance.csv"
    importance_df.to_csv(importance_path, index=False)

    display(importance_df.head(30))
    print("Đã lưu feature importance tại:", importance_path)
else:
    print(f"Model {best_model_name} không có feature_importances_.")

,feature,importance
11,num__electricity_water_interaction,0.337481
5,num__electricity,0.157391
12,num__basic_services_gap,0.118002
6,num__water_access,0.080823
10,num__log1p_gdp_per_capita,0.080012
4,num__sanitation,0.023926
0,num__year,0.013737
2,num__pop_growth,0.009858
50,cat__country_name_Central African Republic,0.007964
7,num__co2_emissions,0.006539


Đã lưu feature importance tại: model_outputs\feature_importance.csv
